# Crypt Analysis
从 slide-level merged mask 中提取 crypt instances，把 crypt 标注加到 adata，做 DE 分析。

In [ ]:
import json
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

## Config

In [ ]:
ADATA_PATH    = "data/Visium_HD_Human_Colon_Cancer_P2/adata.h5ad"   # ← 改成你的路径
MASK_PATH     = "data/Visium_HD_Human_Colon_Cancer_P2/maskann_size-512/sam2_merged_masks/slide_merged_mask_edited.npy"
COORDS_PATH   = "data/Visium_HD_Human_Colon_Cancer_P2/Visium_HD_Human_Colon_Cancer_P2_filtered_coords.npy"
CRYPT_IDS_PATH= "data/Visium_HD_Human_Colon_Cancer_P2/maskann_size-512/sam2_merged_masks/crypt_ids.json"

## Step 1 — Load Data

In [ ]:
adata  = sc.read_h5ad(ADATA_PATH)
mask   = np.load(MASK_PATH)                          # (3072, 3328) uint32
coords = np.load(COORDS_PATH).astype(np.float32)     # (N, 2): col 0 = mask_row, col 1 = mask_col
crypt_ids = set(json.load(open(CRYPT_IDS_PATH)))     # set of int instance IDs

print(f"adata      : {adata.shape}")
print(f"mask       : {mask.shape}  unique instances: {len(np.unique(mask))-1}")
print(f"coords     : {coords.shape}")
print(f"crypt IDs  : {len(crypt_ids)}")
assert coords.shape[0] == adata.n_obs, \
    f"coords rows ({coords.shape[0]}) != adata.n_obs ({adata.n_obs})"

## Step 2 — Map Each Barcode → Mask Instance

In [ ]:
# coords[:,0] = mask row (y),  coords[:,1] = mask col (x)
mask_H, mask_W = mask.shape

rows = np.clip(coords[:, 0].astype(np.int32), 0, mask_H - 1)
cols = np.clip(coords[:, 1].astype(np.int32), 0, mask_W - 1)

instance_ids = mask[rows, cols].astype(np.int32)   # 0 = background

is_crypt     = np.array([i in crypt_ids for i in instance_ids])
in_any_mask  = instance_ids > 0

print(f"Barcodes in any instance : {in_any_mask.sum():,} / {len(instance_ids):,}")
print(f"Barcodes in crypt        : {is_crypt.sum():,}")
print(f"Crypt instances hit      : {len(set(instance_ids[is_crypt]))}  (of {len(crypt_ids)} total crypt IDs)")

## Step 3 — Add Annotations to adata.obs

In [ ]:
adata.obs["mask_instance_id"] = instance_ids
adata.obs["in_mask"]          = in_any_mask
adata.obs["is_crypt"]         = is_crypt
adata.obs["region"]           = np.where(is_crypt, "crypt", "non_crypt")

adata.obs["region"] = pd.Categorical(adata.obs["region"],
                                      categories=["crypt", "non_crypt"])
print(adata.obs["region"].value_counts())

## Step 4 — Preprocessing (skip if already done)

In [ ]:
# 如果 adata 还没有做过 normalize/log，取消注释：
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)

# 如果还没有 PCA / UMAP：
# sc.pp.highly_variable_genes(adata, n_top_genes=2000)
# sc.pp.pca(adata)
# sc.pp.neighbors(adata)
# sc.tl.umap(adata)

print("adata.obs columns:", list(adata.obs.columns))
print("adata.obsm keys  :", list(adata.obsm.keys()))

## Step 5 — UMAP Colored by Crypt Status

In [ ]:
if "X_umap" in adata.obsm:
    sc.pl.umap(adata, color="region",
               palette={"crypt": "#e63946", "non_crypt": "#adb5bd"},
               size=2, title="Crypt vs Non-crypt")
else:
    print("No UMAP found — run sc.tl.umap(adata) first.")

## Step 6 — Spatial Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

# all spots
ax.scatter(coords[~is_crypt, 1], coords[~is_crypt, 0],
           c="#dee2e6", s=0.3, linewidths=0, rasterized=True, label="non_crypt")
# crypt spots
ax.scatter(coords[is_crypt, 1],  coords[is_crypt, 0],
           c="#e63946", s=1.0, linewidths=0, rasterized=True, label="crypt")

ax.set_aspect("equal")
ax.invert_yaxis()          # row 0 at top (image convention)
ax.set_title(f"Crypt barcodes  (n={is_crypt.sum():,})")
ax.legend(markerscale=6, frameon=False)
ax.axis("off")
plt.tight_layout()
plt.show()

## Step 7 — DE Analysis: Crypt vs Non-crypt

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="region",
    groups=["crypt"],
    reference="non_crypt",
    method="wilcoxon",
    key_added="crypt_vs_noncrypt",
)

sc.pl.rank_genes_groups(
    adata, key="crypt_vs_noncrypt",
    n_genes=20, sharey=False
)

In [ ]:
# Top DE genes table
de = sc.get.rank_genes_groups_df(adata, group="crypt", key="crypt_vs_noncrypt")
de_up   = de[de["logfoldchanges"] > 0].sort_values("scores", ascending=False).head(30)
de_down = de[de["logfoldchanges"] < 0].sort_values("scores").head(30)

print("=== Top upregulated in crypt ===")
print(de_up[["names", "logfoldchanges", "pvals_adj", "scores"]].to_string(index=False))
print()
print("=== Top downregulated in crypt ===")
print(de_down[["names", "logfoldchanges", "pvals_adj", "scores"]].to_string(index=False))

In [ ]:
top_genes = de_up["names"].tolist()[:20] + de_down["names"].tolist()[:10]
sc.pl.dotplot(
    adata, var_names=top_genes, groupby="region",
    standard_scale="var", title="Top DE genes: crypt vs non-crypt"
)

## Step 8 — Per-instance Summary (optional)

In [ ]:
# 每个 crypt instance 的 barcode 数量分布
crypt_obs = adata.obs[adata.obs["is_crypt"]]
per_instance = crypt_obs.groupby("mask_instance_id").size().rename("n_barcodes")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(per_instance, bins=30, color="#e63946", edgecolor="none")
ax.set_xlabel("Barcodes per crypt instance")
ax.set_ylabel("Count")
ax.set_title(f"Crypt instance size  (n={len(per_instance)} instances)")
plt.tight_layout()
plt.show()

print(per_instance.describe().round(1))